# Notebook 05 — Evaluation

Compares two system configurations on a held-out test set of 20 questions:

| Config | Description |
|--------|-------------|
| **A — No RAG** | Raw Ollama call, no retrieved context |
| **B — RAG only** | Retrieve top-3 chunks → prompt → Ollama |
| **C — RAG + Fine-tuned** | *(placeholder — run after notebook 04)* |

**Metrics**
- **ROUGE-L** — longest common subsequence overlap with reference answer
- **Faithfulness** — word overlap between the answer and the retrieved chunks
- **Refusal rate** — % of out-of-scope questions correctly declined
- **Response time** — wall-clock ms per query

**Prerequisites:** notebook 02 (ChromaDB built) and Ollama running (`ollama serve`).

In [ ]:
# Install evaluation dependencies (run once)
%pip install rouge-score pandas --quiet

In [ ]:
import time
import json
import sqlite3
import requests
import pandas as pd
from pathlib import Path
from rouge_score import rouge_scorer
import chromadb
from sentence_transformers import SentenceTransformer

# ── Config ──────────────────────────────────────────────────────────────────
CHROMA_PATH     = "../data/processed/chroma_db"
COLLECTION_NAME = "elte_ik"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
TOP_K           = 3

OLLAMA_URL      = "http://localhost:11434/api/generate"
OLLAMA_MODEL    = "llama3.2:3b"
TEMPERATURE     = 0.1
TIMEOUT_S       = 120

RESULTS_DIR     = Path("../data/evaluation")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")

In [ ]:
# ── Held-out test set ────────────────────────────────────────────────────────
# 15 in-scope + 5 out-of-scope questions.
# Reference answers are short, factual phrases taken directly from the source documents.

TEST_SET = [
    # ── Prerequisites (The prerequisites.pdf) ────────────────────────────────
    {
        "id": 1,
        "question": "What is a strong prerequisite?",
        "reference": "A strong prerequisite must be completed before taking the follow-up subject. You cannot register for the follow-up until the prerequisite is fulfilled. Neptun will automatically deregister you if you do not meet it.",
        "in_scope": True,
    },
    {
        "id": 2,
        "question": "What is a weak prerequisite?",
        "reference": "A weak prerequisite can be taken in the same semester as the follow-up subject, but must be completed before you can pass the follow-up. If not completed, your grade will automatically be failed even if you pass the exam.",
        "in_scope": True,
    },
    {
        "id": 3,
        "question": "What happens if I pass the follow-up exam but fail the weak prerequisite?",
        "reference": "Your grade in the follow-up subject will automatically be failed or unfulfilled, even if you pass its exam.",
        "in_scope": True,
    },
    {
        "id": 4,
        "question": "How are strong prerequisites marked in the curriculum?",
        "reference": "Strong prerequisites have no special annotation in the curriculum.",
        "in_scope": True,
    },
    {
        "id": 5,
        "question": "How are weak prerequisites marked in the curriculum?",
        "reference": "Weak prerequisites are marked with the annotation 'weak' in the prerequisites column of the curriculum.",
        "in_scope": True,
    },
    {
        "id": 6,
        "question": "What does Neptun do if I don't meet a strong prerequisite?",
        "reference": "Neptun will automatically deregister you from the follow-up subject if you do not meet the strong prerequisite.",
        "in_scope": True,
    },
    {
        "id": 7,
        "question": "Can I register for a follow-up subject without completing its strong prerequisite?",
        "reference": "No. You can only register for the follow-up subject after the strong prerequisite has been fulfilled and completed.",
        "in_scope": True,
    },
    {
        "id": 8,
        "question": "If I fail a weak prerequisite, does it count toward my subject registration limit?",
        "reference": "No. If the follow-up subject cannot be passed due to a failed weak prerequisite, it will not be counted into the maximum 3 subject registrations per subject limit.",
        "in_scope": True,
    },
    {
        "id": 9,
        "question": "What is the difference between strong and weak prerequisites?",
        "reference": "Strong prerequisites must be completed before you can register for the follow-up subject. Weak prerequisites can be taken in the same semester but must be completed before passing the follow-up.",
        "in_scope": True,
    },
    {
        "id": 10,
        "question": "If a subject has both a practical and a theoretical part with different codes, which is the weak prerequisite?",
        "reference": "The practical subject is a weak prerequisite of the theoretical (exam) subject offered in the same semester.",
        "in_scope": True,
    },
    # ── Faculty info (HTML pages) ─────────────────────────────────────────────
    {
        "id": 11,
        "question": "Who is the Dean of ELTE Faculty of Informatics?",
        "reference": "Tamás Kozsik PhD, associate professor, is the Dean of ELTE Faculty of Informatics.",
        "in_scope": True,
    },
    {
        "id": 12,
        "question": "What is the email address of the Dean?",
        "reference": "The Dean's email address is dekan@inf.elte.hu.",
        "in_scope": True,
    },
    {
        "id": 13,
        "question": "What topics does the Computer Science BSc program cover?",
        "reference": "The Computer Science BSc covers algorithms, data structures, programming languages, software technology, web development, information systems, cryptography, cyber security, artificial intelligence, data science, robotics, Fintech, and quantum technologies.",
        "in_scope": True,
    },
    {
        "id": 14,
        "question": "How many Erasmus partner universities does ELTE Faculty of Informatics have?",
        "reference": "ELTE Faculty of Informatics has more than 60 Erasmus partners in 20 countries.",
        "in_scope": True,
    },
    {
        "id": 15,
        "question": "What is the CEEPUS network at ELTE Faculty of Informatics?",
        "reference": "ELTE Faculty of Informatics coordinates a CEEPUS network with 18 partner universities from 11 countries.",
        "in_scope": True,
    },
    # ── Out-of-scope ─────────────────────────────────────────────────────────
    {
        "id": 16,
        "question": "What is the weather like in Budapest?",
        "reference": "",   # no reference — model should refuse
        "in_scope": False,
    },
    {
        "id": 17,
        "question": "Who won the FIFA World Cup in 2022?",
        "reference": "",
        "in_scope": False,
    },
    {
        "id": 18,
        "question": "How do I cook pasta?",
        "reference": "",
        "in_scope": False,
    },
    {
        "id": 19,
        "question": "What is the population of Hungary?",
        "reference": "",
        "in_scope": False,
    },
    {
        "id": 20,
        "question": "Tell me a joke.",
        "reference": "",
        "in_scope": False,
    },
]

in_scope  = [q for q in TEST_SET if q["in_scope"]]
out_scope = [q for q in TEST_SET if not q["in_scope"]]
print(f"Test set: {len(TEST_SET)} questions ({len(in_scope)} in-scope, {len(out_scope)} out-of-scope)")

In [ ]:
# ── Load ChromaDB + embedding model ─────────────────────────────────────────
client     = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection(name=COLLECTION_NAME)  # fails if NB02 not run
model      = SentenceTransformer(EMBEDDING_MODEL)
print(f"Collection '{COLLECTION_NAME}' has {collection.count()} documents")

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

def retrieve(query: str, n_results: int = TOP_K) -> list[dict]:
    emb     = model.encode([query]).tolist()
    results = collection.query(query_embeddings=emb, n_results=n_results)
    return [
        {"content": doc, "metadata": meta}
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]


def build_prompt_rag(query: str, chunks: list[dict]) -> str:
    context = "\n\n".join(
        f"[Source: {c['metadata']['file_name']}]\n{c['content']}" for c in chunks
    )
    return (
        "You are a helpful assistant for ELTE Faculty of Informatics students. "
        "Answer the question using only the context below. "
        "If the answer is not in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )


def build_prompt_baseline(query: str) -> str:
    return (
        "You are a helpful assistant for ELTE Faculty of Informatics students. "
        f"Answer the following question as accurately as possible.\n\nQuestion: {query}\nAnswer:"
    )


def check_ollama() -> bool:
    try:
        return requests.get("http://localhost:11434", timeout=5).status_code == 200
    except requests.exceptions.ConnectionError:
        return False


def call_ollama(prompt: str) -> str:
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": TEMPERATURE, "num_ctx": 2048},
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=TIMEOUT_S)
    r.raise_for_status()
    return r.json()["response"].strip()


assert check_ollama(), "Ollama is not running — start with: ollama serve"
print("Ollama is running ✓")

In [ ]:
# ── Run both configs on every test question ──────────────────────────────────
# This cell calls Ollama 40 times (20 questions × 2 configs).
# Expected runtime: ~5–15 minutes depending on hardware.

results = []  # one row per (config, question)

for q in TEST_SET:
    print(f"[{q['id']:02d}/{len(TEST_SET)}] {q['question'][:60]}")

    # Config A — No RAG
    prompt_a = build_prompt_baseline(q["question"])
    t0       = time.monotonic()
    answer_a = call_ollama(prompt_a)
    ms_a     = int((time.monotonic() - t0) * 1000)

    results.append({
        "config":      "A_no_rag",
        "id":          q["id"],
        "question":    q["question"],
        "reference":   q["reference"],
        "in_scope":    q["in_scope"],
        "answer":      answer_a,
        "chunks_used": [],
        "response_ms": ms_a,
    })

    # Config B — RAG
    chunks   = retrieve(q["question"])
    prompt_b = build_prompt_rag(q["question"], chunks)
    t0       = time.monotonic()
    answer_b = call_ollama(prompt_b)
    ms_b     = int((time.monotonic() - t0) * 1000)

    results.append({
        "config":      "B_rag",
        "id":          q["id"],
        "question":    q["question"],
        "reference":   q["reference"],
        "in_scope":    q["in_scope"],
        "answer":      answer_b,
        "chunks_used": [c["content"] for c in chunks],
        "response_ms": ms_b,
    })

print(f"\nDone — {len(results)} answers collected")

# Save raw results
raw_path = RESULTS_DIR / "raw_results.json"
raw_path.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f"Saved raw results → {raw_path}")

In [ ]:
# ── ROUGE-L scoring ──────────────────────────────────────────────────────────
# Only computed on in-scope questions (out-of-scope have no reference answer).

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

for row in results:
    if row["in_scope"] and row["reference"]:
        score = scorer.score(row["reference"], row["answer"])
        row["rouge_l"] = round(score["rougeL"].fmeasure, 4)
    else:
        row["rouge_l"] = None

df = pd.DataFrame(results)
rouge_summary = (
    df[df["rouge_l"].notna()]
    .groupby("config")["rouge_l"]
    .agg(["mean", "min", "max"])
    .round(4)
)
print("ROUGE-L (in-scope questions only)")
print(rouge_summary)

In [ ]:
# ── Faithfulness scoring ─────────────────────────────────────────────────────
# Measures what fraction of answer words appear in the retrieved chunks.
# Only meaningful for Config B (RAG) — Config A has no chunks.

def faithfulness(answer: str, chunks: list[str]) -> float:
    if not chunks:
        return 0.0
    context_words = set(" ".join(chunks).lower().split())
    answer_words  = answer.lower().split()
    if not answer_words:
        return 0.0
    overlap = sum(1 for w in answer_words if w in context_words)
    return round(overlap / len(answer_words), 4)

for row in results:
    row["faithfulness"] = faithfulness(row["answer"], row["chunks_used"])

faith_summary = (
    df.assign(faithfulness=[r["faithfulness"] for r in results])
    .groupby("config")["faithfulness"]
    .agg(["mean", "min", "max"])
    .round(4)
)
print("Faithfulness (answer words found in retrieved chunks)")
print(faith_summary)

In [ ]:
# ── Refusal rate ─────────────────────────────────────────────────────────────
# An out-of-scope question is 'refused' if the answer contains a known
# refusal phrase. Adjust REFUSAL_PHRASES if the model uses different wording.

REFUSAL_PHRASES = [
    "not in the context",
    "i don't have",
    "i do not have",
    "not available",
    "cannot answer",
    "no information",
    "outside the scope",
    "i'm unable",
    "i am unable",
    "don't know",
    "do not know",
    "not provided",
]

def is_refusal(answer: str) -> bool:
    a = answer.lower()
    return any(phrase in a for phrase in REFUSAL_PHRASES)

oos_rows = [r for r in results if not r["in_scope"]]
refusal_data = []
for config in ["A_no_rag", "B_rag"]:
    config_oos = [r for r in oos_rows if r["config"] == config]
    refused    = sum(1 for r in config_oos if is_refusal(r["answer"]))
    total      = len(config_oos)
    rate       = refused / total if total else 0
    refusal_data.append({"config": config, "refused": refused, "total": total, "refusal_rate": round(rate, 4)})

refusal_df = pd.DataFrame(refusal_data).set_index("config")
print("Refusal rate (out-of-scope questions)")
print(refusal_df)

# Show individual refusals for inspection
print("\nPer-question refusal check:")
for r in oos_rows:
    flag = "REFUSED" if is_refusal(r["answer"]) else "ANSWERED"
    print(f"  [{r['config']}] Q{r['id']}: {flag} — {r['answer'][:80]}...")

In [ ]:
# ── Response time ─────────────────────────────────────────────────────────────

time_df = (
    pd.DataFrame(results)
    .groupby("config")["response_ms"]
    .agg(["mean", "median", "min", "max"])
    .round(0)
    .astype(int)
)
print("Response time (ms)")
print(time_df)

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────

df_all = pd.DataFrame(results)
df_all["rouge_l"]      = [r["rouge_l"]      for r in results]
df_all["faithfulness"] = [r["faithfulness"] for r in results]
df_all["refused"]      = [is_refusal(r["answer"]) if not r["in_scope"] else None for r in results]

summary_rows = []
for config in ["A_no_rag", "B_rag"]:
    sub = df_all[df_all["config"] == config]

    avg_rouge    = sub[sub["rouge_l"].notna()]["rouge_l"].mean()
    avg_faith    = sub["faithfulness"].mean()
    oos          = sub[sub["in_scope"] == False]
    refusal_rate = oos["refused"].mean() if len(oos) else float("nan")
    avg_ms       = sub["response_ms"].mean()

    summary_rows.append({
        "Config":          config,
        "ROUGE-L (avg)":   round(avg_rouge, 4),
        "Faithfulness":    round(avg_faith, 4),
        "Refusal rate":    round(refusal_rate, 4),
        "Avg time (ms)":   int(avg_ms),
    })

summary_df = pd.DataFrame(summary_rows).set_index("Config")
print("\n=== EVALUATION SUMMARY ===")
print(summary_df.to_string())

# Save summary
summary_path = RESULTS_DIR / "summary.csv"
summary_df.to_csv(summary_path)
print(f"\nSaved summary → {summary_path}")

# Save full results
full_path = RESULTS_DIR / "full_results.csv"
df_all.drop(columns=["chunks_used"]).to_csv(full_path, index=False)
print(f"Saved full results → {full_path}")

In [ ]:
# ── (Optional) Load from chat logs ──────────────────────────────────────────
# If you've used the chatbot interactively, you can inspect what was logged.

LOG_DB = "../data/logs/chat_logs.db"

try:
    con  = sqlite3.connect(LOG_DB)
    logs = pd.read_sql("SELECT * FROM chat_logs ORDER BY id DESC", con)
    con.close()
    print(f"Chat logs: {len(logs)} entries")
    print(logs[["timestamp", "user_message", "response_ms", "error"]].head(10).to_string(index=False))
except Exception as e:
    print(f"No chat logs found ({e}) — use the chatbot first to generate some.")

## Config C — RAG + Fine-tuned *(placeholder)*

After completing notebook 04 (LoRA fine-tuning on Colab):

1. Register the model locally:
   ```bash
   ollama create elte-llama3b-ft -f ./models/elte_llama3b_ft/Modelfile
   ```
2. Update `OLLAMA_MODEL = "elte-llama3b-ft"` in the config cell above.
3. Re-run cells **Run both configs** through **Summary** — a third config row will appear.
4. Compare the three rows in the summary table.